<a href="https://colab.research.google.com/github/minperez/Actividad-5/blob/main/Actividad5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Descripción del Proyecto**

Esta reporte aborda el flujo de trabajo de Machine Learning utilizando el clásico dataset Auto MPG (con información sobre el consumo de combustible y especificaciones de vehículos)

https://archive.ics.uci.edu/dataset/9/auto+mpg

Se comparan dos algoritmos de regresion lineal diferentes (Ridge_Regression vs Random_Forest) utilizando el conjunto de datos Auto MPG (con información sobre el consumo y especificaciones de vehículos).

Para no limitar el estudio a la regresión lineal se incluyo un algoritmo no lineal (Random Forest) para contrastar.

In [4]:
%pip install pandas
%pip install numpy
%pip install scikit-learn
%pip install matplotlib
%pip install seaborn
%pip install mlflow
%pip install scipy

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import os
from scipy import stats
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ==========================================
# 1. Carga, Limpieza y Estandarización
# ==========================================
# Cargar Auto MPG dataset desde UCI
url = "http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
columnas = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
            'acceleration', 'model_year', 'origin', 'car_name']
df = pd.read_csv(url, names=columnas, na_values='?', comment='\t', sep=' ', skipinitialspace=True)

# Limpieza
df = df.drop(columns=['car_name']) # Eliminamos variable categórica no predictiva
df['horsepower'] = df['horsepower'].fillna(df['horsepower'].median()) # Imputación de nulos

# Separar variables predictoras (X) y objetivo (y)
X = df.drop(columns=['mpg'])
y = df['mpg']

# Estandarización
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# ==========================================
# 2. Identificación de Patrones y Anomalías (EDA Artifacts)
# ==========================================
os.makedirs("artefactos", exist_ok=True)

# Mapa de correlación
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Matriz de Correlación - Auto MPG")
plt.savefig("artefactos/correlacion.png")
plt.close()

# Boxplots para detección de anomalías
plt.figure(figsize=(12, 6))
sns.boxplot(data=X)
plt.title("Boxplots de Variables (Detección de Outliers)")
plt.savefig("artefactos/anomalias.png")
plt.close()

# ==========================================
# 3. Configuración de Modelos y MLflow
# ==========================================
mlflow.set_experiment("Analisis_Auto_MPG")

def entrenar_y_registrar(nombre_modelo, modelo, param_grid, X_train, y_train, X_test, y_test):
    with mlflow.start_run(run_name=nombre_modelo):
        # Grid Search con validación cruzada (k-fold = 5)
        grid_search = GridSearchCV(estimator=modelo, param_grid=param_grid,
                                   cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
        grid_search.fit(X_train, y_train)

        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test)

        # Métricas
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        # Guardar CV scores para comparación estadística posterior
        cv_scores = cross_val_score(best_model, X_train, y_train, cv=10, scoring='neg_mean_squared_error')

        # Registro en MLflow
        mlflow.log_params(grid_search.best_params_)
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2", r2)

        # Registro del modelo y artefactos
        mlflow.sklearn.log_model(best_model, f"modelo_{nombre_modelo}")
        mlflow.log_artifacts("artefactos", artifact_path="eda_plots")

        print(f"--- Resultados para {nombre_modelo} ---")
        print(f"Mejores Parámetros: {grid_search.best_params_}")
        print(f"MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}\n")

        return cv_scores, best_model

# Modelo 1: Regresión Ridge (Regresión Lineal con regularización L2)
ridge_params = {'alpha': [0.1, 1.0, 10.0, 50.0, 100.0]}
ridge_cv_scores, modelo_ridge = entrenar_y_registrar(
    "Ridge_Regression", Ridge(), ridge_params, X_train, y_train, X_test, y_test
)

# Modelo 2: Random Forest
rf_params = {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20], 'min_samples_split': [2, 5]}
rf_cv_scores, modelo_rf = entrenar_y_registrar(
    "Random_Forest", RandomForestRegressor(random_state=42), rf_params, X_train, y_train, X_test, y_test
)

# ==========================================
# 4. Comparación Estadística
# ==========================================
# Usamos un T-Test pareado sobre los resultados negativos del MSE en la validación cruzada (10 folds)
t_stat, p_value = stats.ttest_rel(ridge_cv_scores, rf_cv_scores)

print("--- Comparación Estadística (T-Test Pareado sobre CV MSE) ---")
print(f"Estadístico T: {t_stat:.4f}")
print(f"P-Valor: {p_value:.6f}")
if p_value < 0.05:
    print("Conclusión: Existe una diferencia estadísticamente significativa entre el rendimiento de ambos modelos.")
else:
    print("Conclusión: NO existe una diferencia estadísticamente significativa entre los modelos.")

2026/06/29 02:20:23 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/29 02:20:23 INFO mlflow.store.db.utils: Updating database tables
2026/06/29 02:20:26 INFO mlflow.tracking.fluent: Experiment with name 'Analisis_Auto_MPG' does not exist. Creating a new experiment.
2026/06/29 02:20:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


--- Resultados para Ridge_Regression ---
Mejores Parámetros: {'alpha': 1.0}
MAE: 2.2513 | RMSE: 2.8648 | R²: 0.8474



2026/06/29 02:21:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


--- Resultados para Random_Forest ---
Mejores Parámetros: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200}
MAE: 1.5787 | RMSE: 2.1865 | R²: 0.9111

--- Comparación Estadística (T-Test Pareado sobre CV MSE) ---
Estadístico T: -3.0124
P-Valor: 0.014660
Conclusión: Existe una diferencia estadísticamente significativa entre el rendimiento de ambos modelos.
